#  FAISS Tuning Notebook

In [ ]:
# ==== Config + env ====
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_FILE = '.env'   # change to '.env.local' if needed
EMBED_LIMIT = 5000
DATASET_FILTER = None   # e.g. 'kaggle'
RANDOM_SEED = 42
MAX_DOWNLOAD_WORKERS = int(os.getenv('EMBED_DOWNLOAD_WORKERS', '16'))
DOWNLOAD_RETRIES = int(os.getenv('EMBED_DOWNLOAD_RETRIES', '2'))
EMBED_MODEL = os.getenv('EMBED_MODEL', 'google/siglip2-base-patch16-naflex')
EMBED_DEVICE = os.getenv('EMBED_DEVICE', 'cpu')


def find_backend_api_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if p.name == 'backend-api':
            return p
        candidate = p / 'backend-api'
        if candidate.exists() and candidate.is_dir():
            return candidate
    raise RuntimeError('Could not locate backend-api root')


BACKEND_API_ROOT = find_backend_api_root(Path.cwd())
ENV_PATH = BACKEND_API_ROOT / ENV_FILE
load_dotenv(ENV_PATH, override=False)

DB_DSN = os.getenv('DB_DSN') or os.getenv('DB_URL')
if not DB_DSN:
    raise RuntimeError('Missing DB_DSN/DB_URL in env')

S3_ENDPOINT_URL = os.getenv('S3_ENDPOINT_URL')
S3_REGION = os.getenv('S3_REGION', 'us-east-1')
S3_ACCESS_KEY_ID = os.getenv('S3_ACCESS_KEY_ID')
S3_SECRET_ACCESS_KEY = os.getenv('S3_SECRET_ACCESS_KEY')
S3_BUCKET = os.getenv('S3_BUCKET')
S3_FORCE_PATH_STYLE = str(os.getenv('S3_FORCE_PATH_STYLE', 'true')).lower() in {'1','true','yes','on'}

if not all([S3_ENDPOINT_URL, S3_ACCESS_KEY_ID, S3_SECRET_ACCESS_KEY, S3_BUCKET]):
    raise RuntimeError('Missing S3 env vars (endpoint/key/secret/bucket)')

print('env file:', ENV_PATH)
print('db dsn prefix:', DB_DSN.split('@')[0])
print('s3 endpoint:', S3_ENDPOINT_URL)
print('s3 bucket:', S3_BUCKET)
print('download workers:', MAX_DOWNLOAD_WORKERS)
print('download retries:', DOWNLOAD_RETRIES)

print('embed model:', EMBED_MODEL)
print('embed device:', EMBED_DEVICE)


In [ ]:
%pip install -q faiss-cpu numpy pandas matplotlib pillow boto3 psycopg2-binary python-dotenv tqdm transformers torch


In [ ]:
# ==== Load embeddings + metadata from Postgres/S3 (concurrent downloads) ====
import io
import json
import random
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import UTC, datetime

import boto3
import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg2
from PIL import Image
from tqdm.auto import tqdm

plt.rcParams['figure.figsize'] = (14, 8)
pd.set_option('display.max_colwidth', 140)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

conn = psycopg2.connect(DB_DSN)
conn.autocommit = False


def make_s3_client():
    return boto3.client(
        's3',
        endpoint_url=S3_ENDPOINT_URL,
        region_name=S3_REGION,
        aws_access_key_id=S3_ACCESS_KEY_ID,
        aws_secret_access_key=S3_SECRET_ACCESS_KEY,
        config=boto3.session.Config(s3={'addressing_style': 'path' if S3_FORCE_PATH_STYLE else 'auto'}),
    )


s3 = make_s3_client()


def load_embedding_from_s3(client, key: str) -> np.ndarray:
    obj = client.get_object(Bucket=S3_BUCKET, Key=key)
    body = obj['Body'].read()
    return np.load(io.BytesIO(body)).astype(np.float32).reshape(-1)


def fetch_one(index_and_row):
    idx, row = index_and_row
    key = row['embed_s3_key']
    client = make_s3_client()
    last_exc = None
    for _ in range(max(1, DOWNLOAD_RETRIES + 1)):
        try:
            emb = load_embedding_from_s3(client, key)
            return idx, row, emb, None
        except Exception as exc:
            last_exc = exc
    return idx, row, None, str(last_exc)


def load_data(limit: int | None = None, dataset: str | None = None):
    sql = '''
        SELECT
            i.id AS image_id,
            i.sha256,
            i.s3_key,
            i.dataset,
            i.width,
            i.height,
            p.embed_s3_key,
            a.caption_text,
            a.ocr_text
        FROM images i
        JOIN processing p ON p.image_id = i.id
        LEFT JOIN annotations a ON a.image_id = i.id
        WHERE p.embed_status = 'DONE'
          AND p.embed_s3_key IS NOT NULL
          {dataset_clause}
        ORDER BY i.id ASC
        {limit_clause}
    '''
    dataset_clause = "AND i.dataset = %(dataset)s" if dataset else ""
    limit_clause = f"LIMIT {int(limit)}" if limit else ""
    sql = sql.format(dataset_clause=dataset_clause, limit_clause=limit_clause)

    with conn.cursor() as cur:
        cur.execute(sql, {'dataset': dataset} if dataset else {})
        rows = cur.fetchall()

    cols = ['image_id', 'sha256', 's3_key', 'dataset', 'width', 'height', 'embed_s3_key', 'caption', 'ocr_text']
    raw_df = pd.DataFrame(rows, columns=cols)
    records = raw_df.to_dict(orient='records')

    ok = []
    failed = 0
    workers = max(1, min(MAX_DOWNLOAD_WORKERS, len(records) or 1))
    print(f'downloading {len(records)} embeddings with workers={workers}...')

    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = [ex.submit(fetch_one, x) for x in enumerate(records)]
        for fut in tqdm(as_completed(futs), total=len(futs), desc='Embedding downloads'):
            idx, row, emb, err = fut.result()
            if emb is None:
                failed += 1
                continue
            ok.append((idx, row, emb))

    if not ok:
        raise RuntimeError('No embeddings loaded from DB/S3')

    ok.sort(key=lambda x: x[0])
    keep_rows = [r for _, r, _ in ok]
    vectors = [e for _, _, e in ok]

    X = np.stack(vectors).astype(np.float32)
    meta_df = pd.DataFrame(keep_rows).drop(columns=['embed_s3_key'])
    print({'loaded': len(vectors), 'failed': failed, 'dimension': int(X.shape[1])})
    return X, meta_df


X, meta_df = load_data(limit=EMBED_LIMIT, dataset=DATASET_FILTER)
print('embeddings shape:', X.shape)
display(meta_df.head(5))


In [ ]:
# ==== FAISS helpers: build/search/save/list/swap/load ====
INDEX_DIR = BACKEND_API_ROOT / 'data' / 'cache' / 'tuning_indexes'
INDEX_DIR.mkdir(parents=True, exist_ok=True)
ACTIVE_FILE = INDEX_DIR / 'ACTIVE_VERSION'


def normalize(x: np.ndarray) -> np.ndarray:
    y = np.ascontiguousarray(x.astype(np.float32))
    faiss.normalize_L2(y)
    return y


def build_index(x: np.ndarray, kind: str, **params):
    x = normalize(x)
    n, d = x.shape

    if kind == 'flat':
        idx = faiss.IndexFlatIP(d)
    elif kind == 'ivf':
        nlist = int(params.get('nlist', min(4096, max(64, int(np.sqrt(n))))))
        q = faiss.IndexFlatIP(d)
        idx = faiss.IndexIVFFlat(q, d, nlist, faiss.METRIC_INNER_PRODUCT)
    elif kind == 'hnsw':
        m = int(params.get('m', 32))
        idx = faiss.IndexHNSWFlat(d, m, faiss.METRIC_INNER_PRODUCT)
        idx.hnsw.efConstruction = int(params.get('efConstruction', 120))
    elif kind == 'ivfpq':
        nlist = int(params.get('nlist', min(4096, max(64, int(np.sqrt(n))))))
        m = int(params.get('m', 16))
        nbits = int(params.get('nbits', 8))
        q = faiss.IndexFlatIP(d)
        idx = faiss.IndexIVFPQ(q, d, nlist, m, nbits, faiss.METRIC_INNER_PRODUCT)
    else:
        raise ValueError(f'Unknown kind: {kind}')

    if not idx.is_trained:
        idx.train(x)
    idx.add(x)
    return idx


def set_search_params(idx, **params):
    ps = faiss.ParameterSpace()
    for k, v in params.items():
        try:
            ps.set_index_parameter(idx, str(k), float(v))
        except Exception:
            pass


def save_version(idx, image_ids: list[int], kind: str, build_params: dict):
    version = f"{kind}-v{datetime.now(UTC).strftime('%Y%m%d-%H%M%S')}"
    vdir = INDEX_DIR / version
    vdir.mkdir(parents=True, exist_ok=True)

    faiss.write_index(idx, str(vdir / 'index.faiss'))
    (vdir / 'mapping.json').write_text(json.dumps({i: int(img_id) for i, img_id in enumerate(image_ids)}))
    (vdir / 'metadata.json').write_text(json.dumps({
        'version': version,
        'kind': kind,
        'build_params': build_params,
        'num_vectors': int(idx.ntotal),
        'dimension': int(idx.d),
        'created_at': datetime.now(UTC).isoformat(),
    }, indent=2))
    return version


def list_versions() -> pd.DataFrame:
    out = []
    for d in sorted(INDEX_DIR.glob('*')):
        if not d.is_dir() or not (d / 'metadata.json').exists():
            continue
        m = json.loads((d / 'metadata.json').read_text())
        m['version'] = d.name
        out.append(m)
    return pd.DataFrame(out).sort_values('created_at', ascending=False) if out else pd.DataFrame()


def swap_to_version(version: str):
    vdir = INDEX_DIR / version
    if not vdir.exists():
        raise FileNotFoundError(f'Version not found: {version}')
    ACTIVE_FILE.write_text(version)


def get_active_version() -> str | None:
    return ACTIVE_FILE.read_text().strip() if ACTIVE_FILE.exists() else None


def load_version(version: str):
    vdir = INDEX_DIR / version
    idx = faiss.read_index(str(vdir / 'index.faiss'))
    mapping = {int(k): int(v) for k, v in json.loads((vdir / 'mapping.json').read_text()).items()}
    meta = json.loads((vdir / 'metadata.json').read_text())
    return idx, mapping, meta


In [ ]:
# ==== Build and evaluate index families (flat/ivf/hnsw/ivfpq) ====
Xn = normalize(X)
image_ids = meta_df['image_id'].astype(int).tolist()
N, D = Xn.shape
print({'num_vectors': N, 'dim': D})

# FAISS heuristic: training k centroids is more stable with ~39*k points.
# For small datasets, large nlist/nbits creates warnings and poor training quality.
def max_reasonable_k(n_vectors: int) -> int:
    return max(1, n_vectors // 39)


def choose_nlist_candidates(n_vectors: int) -> list[int]:
    kmax = max_reasonable_k(n_vectors)
    base = [8, 16, 32, 64, 128, 256, 512]
    out = [k for k in base if k <= kmax]
    if not out:
        out = [max(1, min(8, n_vectors // 20))]
    return sorted(set(out))


def choose_pq_nbits(n_vectors: int) -> int:
    # Need ~39 * 2^nbits points for stable codebook training.
    for b in [8, 7, 6, 5, 4]:
        if n_vectors >= 39 * (2 ** b):
            return b
    return 4


nlist_candidates = choose_nlist_candidates(N)
pq_nbits = choose_pq_nbits(N)

if max(nlist_candidates) < 64:
    print(
        f"small dataset detected (N={N}): using nlist={nlist_candidates} and PQ nbits={pq_nbits} to avoid unstable IVF/PQ training"
    )

baseline = build_index(Xn, 'flat')

configs = [('flat', {})]
for nlist in nlist_candidates[-2:]:  # keep runtime reasonable
    configs.append(('ivf', {'nlist': int(nlist)}))
configs.append(('hnsw', {'m': 32, 'efConstruction': 120}))
configs.append(('ivfpq', {'nlist': int(nlist_candidates[-1]), 'm': 16, 'nbits': int(pq_nbits)}))

qn = min(200, len(Xn))
qidx = np.random.choice(len(Xn), size=qn, replace=False)
Q = Xn[qidx]
K = 20
_, I_ref = baseline.search(Q, K)

eval_rows = []
saved = []

for kind, build_params in configs:
    idx = build_index(Xn, kind, **build_params)

    param_sweep = [{}]
    if kind in {'ivf', 'ivfpq'}:
        nlist = int(build_params['nlist'])
        probes = [4, 8, 16, 32, 64]
        probes = [p for p in probes if p <= nlist]
        param_sweep = [{'nprobe': p} for p in probes] or [{'nprobe': 1}]
    elif kind == 'hnsw':
        param_sweep = [{'efSearch': 32}, {'efSearch': 64}, {'efSearch': 128}]

    for sp in param_sweep:
        set_search_params(idx, **sp)

        t0 = time.perf_counter()
        Dk, Ik = idx.search(Q, K)
        dt = (time.perf_counter() - t0) * 1000.0
        ms_per_query = dt / len(Q)

        overlap = []
        for a, b in zip(I_ref, Ik):
            sa = set(int(x) for x in a if x >= 0)
            sb = set(int(x) for x in b if x >= 0)
            overlap.append(len(sa & sb) / max(1, len(sa)))

        eval_rows.append({
            'kind': kind,
            'build_params': json.dumps(build_params, sort_keys=True),
            'search_params': json.dumps(sp, sort_keys=True),
            'recall_at_k': round(float(np.mean(overlap)), 4),
            'ms_per_query': round(float(ms_per_query), 4),
            'queries': len(Q),
            'k': K,
        })

    version = save_version(idx, image_ids=image_ids, kind=kind, build_params=build_params)
    saved.append({'version': version, 'kind': kind, 'build_params': build_params})

results_df = pd.DataFrame(eval_rows).sort_values(['recall_at_k', 'ms_per_query'], ascending=[False, True])
display(results_df)
print('saved versions:')
display(pd.DataFrame(saved))
print('all versions:')
display(list_versions())


In [ ]:
# ==== Query controls: compare results across index types/versions ====
# Query modes:
# - 'image_id': use an existing image's embedding as query
# - 'vector_file': load query vector from local .npy file
# - 'text': encode text query locally with SigLIP (transformers)
QUERY_MODE = 'text'
QUERY_IMAGE_ID = int(meta_df.iloc[0]['image_id'])
QUERY_VECTOR_FILE = None  # e.g. '/tmp/query_vec.npy'
TEXT_QUERY = 'drake meme'
TOP_K = 12

# Which versions to compare:
# - None => latest per kind
# - list[str] => exact versions
SELECTED_VERSIONS = None

# Per-kind search params used during search
SEARCH_PARAMS_BY_KIND = {
    'flat': {},
    'ivf': {'nprobe': 32},
    'hnsw': {'efSearch': 128},
    'ivfpq': {'nprobe': 32},
}


_TEXT_ENCODER = None

def get_text_encoder():
    global _TEXT_ENCODER
    if _TEXT_ENCODER is not None:
        return _TEXT_ENCODER

    import torch
    from transformers import AutoModel, AutoProcessor

    model_name = EMBED_MODEL
    device = 'cuda' if EMBED_DEVICE == 'cuda' and torch.cuda.is_available() else 'cpu'

    processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModel.from_pretrained(model_name, trust_remote_code=True)
    model = model.to(device)
    model.eval()

    is_siglip2 = 'siglip2' in model_name.lower()
    has_get_text_features = hasattr(model, 'get_text_features')

    def encode(text: str) -> np.ndarray:
        t = text.lower() if is_siglip2 else text
        kwargs = {'text': [t], 'return_tensors': 'pt', 'padding': 'max_length'}
        if is_siglip2:
            kwargs['max_length'] = 64
        inputs = processor(**kwargs)

        new_inputs = {}
        for k, v in inputs.items():
            new_inputs[k] = v.to(device)

        with torch.no_grad():
            if has_get_text_features:
                feats = model.get_text_features(**new_inputs)
            else:
                out = model(**new_inputs)
                if hasattr(out, 'text_embeds'):
                    feats = out.text_embeds
                elif hasattr(out, 'pooler_output'):
                    feats = out.pooler_output
                elif hasattr(out, 'last_hidden_state'):
                    feats = out.last_hidden_state[:, 0, :]
                else:
                    raise RuntimeError('Unknown model output format for text features')

        v = feats.detach().cpu().numpy().astype(np.float32)
        faiss.normalize_L2(v)
        return v

    _TEXT_ENCODER = encode
    return _TEXT_ENCODER


def choose_versions(df: pd.DataFrame, selected=None):
    if df.empty:
        return []
    if selected:
        return [v for v in selected if v in set(df['version'])]

    chosen = []
    for kind in ['flat', 'ivf', 'hnsw', 'ivfpq']:
        sub = df[df['kind'] == kind]
        if not sub.empty:
            chosen.append(sub.iloc[0]['version'])
    return chosen


def get_query_vector() -> np.ndarray:
    if QUERY_MODE == 'image_id':
        id_to_pos_full = {int(img_id): i for i, img_id in enumerate(meta_df['image_id'].astype(int).tolist())}
        if QUERY_IMAGE_ID not in id_to_pos_full:
            raise ValueError(f'QUERY_IMAGE_ID not found in loaded data: {QUERY_IMAGE_ID}')
        return Xn[id_to_pos_full[QUERY_IMAGE_ID]].reshape(1, -1)

    if QUERY_MODE == 'vector_file':
        if not QUERY_VECTOR_FILE:
            raise ValueError('Set QUERY_VECTOR_FILE when QUERY_MODE=vector_file')
        v = np.load(QUERY_VECTOR_FILE).astype(np.float32).reshape(1, -1)
        faiss.normalize_L2(v)
        return v

    if QUERY_MODE == 'text':
        if not TEXT_QUERY or not TEXT_QUERY.strip():
            raise ValueError('Set TEXT_QUERY when QUERY_MODE=text')
        encode = get_text_encoder()
        return encode(TEXT_QUERY)

    raise ValueError('Unknown QUERY_MODE')


def preview_from_s3(df: pd.DataFrame, title: str, cols: int = 4):
    if df.empty:
        print(f'{title}: no hits')
        return

    rows_n = int(np.ceil(len(df) / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(4.8 * cols, 4.2 * rows_n))
    fig.suptitle(title)
    axes = np.array(axes).reshape(-1)

    for i, ax in enumerate(axes):
        if i >= len(df):
            ax.axis('off')
            continue

        r = df.iloc[i]
        key = r.get('s3_key')
        try:
            if not key:
                raise RuntimeError('missing s3_key')
            obj = s3.get_object(Bucket=S3_BUCKET, Key=key)
            img = Image.open(io.BytesIO(obj['Body'].read())).convert('RGB')
            ax.imshow(img)
        except Exception as exc:
            ax.text(0.5, 0.5, f'preview failed\n{exc}', ha='center', va='center')

        ax.set_title(f"id={int(r['image_id'])} score={float(r['score']):.4f}")
        ax.axis('off')

    plt.tight_layout()
    plt.show()


versions_df = list_versions()
if versions_df.empty:
    raise RuntimeError('No saved versions. Run build/eval cell first.')

target_versions = choose_versions(versions_df, SELECTED_VERSIONS)
if not target_versions:
    raise RuntimeError('No versions selected to compare.')

qvec = get_query_vector()
if qvec.shape[1] != Xn.shape[1]:
    raise ValueError(f'Query dim {qvec.shape[1]} does not match index dim {Xn.shape[1]}. Use same EMBED_MODEL as indexed embeddings.')

all_tables = []

for version in target_versions:
    idx, mapping, idx_meta = load_version(version)
    kind = idx_meta.get('kind', 'unknown')
    params = SEARCH_PARAMS_BY_KIND.get(kind, {})
    set_search_params(idx, **params)

    t0 = time.perf_counter()
    D, I = idx.search(qvec, TOP_K + 1)
    t_ms = (time.perf_counter() - t0) * 1000.0

    pos_to_id = mapping
    rows = []
    for pos, score in zip(I[0], D[0]):
        if pos < 0:
            continue
        img_id = pos_to_id.get(int(pos))
        if img_id is None:
            continue
        if QUERY_MODE == 'image_id' and img_id == QUERY_IMAGE_ID:
            continue

        hit = meta_df.loc[meta_df['image_id'] == img_id].head(1)
        if hit.empty:
            continue

        rec = hit.iloc[0].to_dict()
        rec['score'] = float(score)
        rec['version'] = version
        rec['kind'] = kind
        rec['latency_ms'] = round(t_ms, 3)
        rec['query_mode'] = QUERY_MODE
        rec['text_query'] = TEXT_QUERY if QUERY_MODE == 'text' else None
        rows.append(rec)

    hits_df = pd.DataFrame(rows).head(TOP_K)
    if not hits_df.empty:
        display(hits_df[['version', 'kind', 'query_mode', 'latency_ms', 'image_id', 'score', 'dataset', 'caption']])
        preview_from_s3(hits_df, title=f"{version} ({kind})", cols=4)
        all_tables.append(hits_df)

if all_tables:
    combined = pd.concat(all_tables, ignore_index=True)
    print('combined top hits (first 40 rows):')
    display(combined[['version', 'kind', 'query_mode', 'latency_ms', 'image_id', 'score', 'dataset']].head(40))
